# NB0: Setup, Motivación y Ley de Amdahl

**Computación de Altas Prestaciones para Ciencia de Datos (CAPCD)**

---

## Objetivos de este notebook

1. Configurar el entorno de Google Colab con GPU.
2. Entender por qué Python es lento y qué haremos al respecto.
3. Comprender la **Ley de Amdahl** y sus implicaciones prácticas.

**Tiempo estimado:** 30 minutos

## 1. ¿Por qué este curso?

Python es el lenguaje dominante en Ciencia de Datos, Machine Learning e IA. Pero tiene un problema: es lento.

Un bucle `for` en Python puro puede ser de 100 a 1000 veces más lento que el mismo bucle en C/C. Cuando trabajamos con millones de datos, esto importa.

### La buena noticia

No necesitamos abandonar Python. Existen herramientas que nos permiten:

| Estrategia | Herramienta | Speedup típico |
|---|---|---|
| Medir dónde está el cuello de botella | `line_profiler`, `memory_profiler` | — |
| Usar múltiples núcleos de CPU | `concurrent.futures` | 2–16x |
| Compilar Python a código máquina | `Numba @njit` | 50–200x |
| DataFrames ultrarrápidos en CPU | `Polars` | 5–50x |
| Ejecutar en miles de núcleos GPU | `CuPy`, `Numba @cuda.jit` | 100–1000x |
| Data Science completa en GPU | `RAPIDS` (`cuDF`, `cuML`) | 10–100x |

Este curso recorre esas estrategias, de menor a mayor potencia.

## Demo 1: Activar GPU en Google Colab

Antes de empezar, necesitamos activar el runtime con GPU:

1. Ve a la esquina superior derecha y pulsa en *Additional connection options*.
2. Selecciona *Change runtime type*.
3. Elige **Python 3** y **T4 GPU**.
3. Haz clic en *Save* y después en *Reconnect*.

Verifica ejecutando la siguiente celda:

In [ ]:
# Verificar que tenemos GPU disponible
!nvidia-smi

Si ves información sobre una GPU (Tesla T4 o similar), ¡estás listo! Si no, revisa que hayas seleccionado GPU en el runtime.

## Demo 2: Comprobación de las dependencias

Aunque el entorno de Colab ya trae la mayoría de librerías populares de ciencia de datos preinstaladas, en alguno de los notebooks necesitaremos ejecutar un `pip install` mínimo para instalar algunas librerías particulares. Esto se debe repetir cada vez que se reinicie el runtime de Colab.

In [ ]:
# Verificar instalación
import numpy as np
import numba
import cupy as cp

print(f"NumPy:  {np.__version__}")
print(f"Numba:  {numba.__version__}")
print(f"CuPy:   {cp.__version__}")
print(f"\n✅ Todo instalado correctamente")

## **Importante:** Guardando vuestro progreso en los notebooks.

Si habéis abierto este notebook en Colab desde el enlace del curso, veréis que arriba hay un icono de una nube tachada con el mensaje *Save in GitHub to keep changes*. Eso significa que Colab está abriendo este notebook directamente desde mi repositorio de GitHub y los cambios que hagáis no se guardarán si cerráis la ventana.

Os recomiendo que, cada vez que comencéis un nuevo notebook, pulséis el botón que hay justo debajo: *Copy to Drive*. Así haréis una copia en vuestro Google Drive que podréis modificar y guardar. Podéis cambiarle el nombre a ese nuevo notebook para no mezclarlo con el original, aparecerá con un icono de Google Drive antes del título.

## 2. ¿Por qué Python es lento?

Python es un lenguaje **interpretado** y con **tipado dinámico**. Cada operación simple implica:

1. Buscar el tipo del objeto en runtime
2. Buscar el método correcto para ese tipo
3. Ejecutar la operación
4. Crear un nuevo objeto con el resultado

En C, una suma de enteros es **una instrucción de CPU**. En Python, es una cadena de operaciones con overhead.

### Veámoslo con números

In [ ]:
import time
import numpy as np

N = 10_000_000

# --- Python puro ---
data = list(range(N))
start = time.perf_counter()
result_python = sum(x * x for x in data)
t_python = time.perf_counter() - start
print(f"Python puro:  {t_python:.3f}s")

# --- NumPy (vectorizado) ---
data_np = np.arange(N)
start = time.perf_counter()
result_numpy = np.sum(data_np * data_np)
t_numpy = time.perf_counter() - start
print(f"NumPy:        {t_numpy:.3f}s")

print(f"\nSpeedup NumPy vs Python: {t_python / t_numpy:.1f}x")

NumPy ya es mucho más rápido porque internamente usa C. Pero a lo largo del curso veremos cómo ir aún más lejos: compilando con Numba y ejecutando en GPU.

## 3. La Ley de Amdahl

La Ley de Amdahl nos dice cuál es el *speedup* máximo teórico al paralelizar un programa:

$$S(n) = \frac{1}{(1 - P) + \frac{P}{n}}$$

Donde:
- $P$ = Fracción del código que se puede paralelizar (0 a 1)
- $n$ = Número de núcleos utilizados
- $S(n)$ = *Speedup* obtenido

### Implicación clave

Si solo el 50% de tu código es paralelizable ($P=0.5$), no importa cuántos núcleos tengas: el *speedup* máximo es **2x**.

$$S(\infty) = \frac{1}{1 - P}$$

Por eso el *profiling* (Notebook 1) es tan importante: necesitamos saber dónde está el cuello de botella antes de optimizar.

## Demo 3: Visualizar la Ley de Amdahl

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

n_cores = np.arange(1, 129)
P_values = [0.5, 0.75, 0.9, 0.95, 0.99]

plt.figure(figsize=(10, 6))
for P in P_values:
    speedup = 1 / ((1 - P) + P / n_cores)
    plt.plot(n_cores, speedup, label=f"P = {P}")

plt.xlabel("Número de núcleos (n)")
plt.ylabel("Speedup S(n)")
plt.title("Ley de Amdahl: Speedup según fracción paralelizable")
plt.legend()
plt.grid(True, alpha=0.3)
plt.xlim(1, 128)
plt.show()

### Lectura del gráfico

- Con $P=0.5$, ni con 128 núcleos pasamos de 2x: no vale la pena paralelizar más.
- Con $P=0.95$, podemos llegar a ~20x con suficientes núcleos.
- Con $P=0.99$, una GPU con miles de núcleos sí marca diferencia.

**Moraleja:** Antes de lanzarte a paralelizar, mide cuánto de tu código es realmente el cuello de botella.

---

## Ejercicio 1: Calcula el speedup teórico

Tienes un programa donde el 80% del tiempo se gasta en una función paralelizable. Calcula el speedup teórico con 4, 8 y 16 núcleos usando la Ley de Amdahl.

In [ ]:
def amdahl_speedup(P, n):
    """Calcula el speedup según la Ley de Amdahl.

    Args:
        P: Fracción paralelizable (entre 0 y 1)
        n: Número de núcleos

    Returns:
        Speedup teórico
    """
    # TODO: Implementa la fórmula de Amdahl
    pass


P = 0.8
for n in [4, 8, 16]:
    s = amdahl_speedup(P, n)
    print(f"P={P}, n={n:>2d} → Speedup = {s:.2f}x")

In [ ]:
# ✅ Autoevaluación
assert abs(amdahl_speedup(0.8, 4) - 2.5) < 0.01, "Speedup con 4 núcleos debe ser ~2.5"
assert abs(amdahl_speedup(0.8, 8) - 3.33) < 0.01, "Speedup con 8 núcleos debe ser ~3.33"
assert abs(amdahl_speedup(0.0, 1000) - 1.0) < 0.01, "Si nada es paralelizable, speedup = 1"
assert abs(amdahl_speedup(1.0, 4) - 4.0) < 0.01, "Si todo es paralelizable, speedup = n"
print("✅ ¡Todos los tests pasados!")

---

## Ejercicio 2: Python vs NumPy

Escribe una función en Python puro que calcule la media de una lista de números, y compárala con `np.mean()`. Mide los tiempos con `time.perf_counter()`.

In [ ]:
import time
import numpy as np

N = 5_000_000
data_list = list(range(N))
data_array = np.arange(N, dtype=np.float64)


def media_python(datos):
    """Calcula la media de una lista usando solo Python puro."""
    # TODO: Implementar sin usar numpy ni statistics
    pass


# Medir Python puro
start = time.perf_counter()
resultado_python = media_python(data_list)
t_python = time.perf_counter() - start

# Medir NumPy
start = time.perf_counter()
resultado_numpy = np.mean(data_array)
t_numpy = time.perf_counter() - start

print(f"Python puro: {t_python:.4f}s  (resultado: {resultado_python})")
print(f"NumPy:       {t_numpy:.4f}s  (resultado: {resultado_numpy})")
print(f"Speedup:     {t_python / t_numpy:.1f}x")

In [ ]:
# ✅ Autoevaluación
assert resultado_python is not None, "La función no debe devolver None"
assert abs(resultado_python - resultado_numpy) < 1.0, "Los resultados deben coincidir"
assert t_python > t_numpy, "NumPy debería ser más rápido que Python puro"
print("✅ ¡Correcto! NumPy es significativamente más rápido.")